In [ ]:
#!/usr/bin/env python3
"""
TCGA Data Download Script for Gene Mutations
Downloads mutation data and clinical data from TCGA via GDC API
"""

import requests
import json
import pandas as pd
import os
import time
from typing import List, Dict, Optional
import argparse
from pathlib import Path

class TCGADownloader:
    def __init__(self, output_dir: str = "tcga_data"):
        self.base_url = "https://api.gdc.cancer.gov"
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
    def get_projects(self) -> List[Dict]:
        """Get all available TCGA projects"""
        endpoint = f"{self.base_url}/projects"
        params = {
            "filters": json.dumps({
                "op": "in",
                "content": {
                    "field": "program.name",
                    "value": ["TCGA"]
                }
            }),
            "format": "json",
            "size": "2000"
        }
        
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        
        # Debug the response structure
        json_response = response.json()
        print(f"Files API response keys: {json_response.keys()}")
        if 'data' in json_response:
            print(f"Data keys: {json_response['data'].keys()}")
            if 'hits' in json_response['data']:
                print(f"Number of hits: {len(json_response['data']['hits'])}")
                if json_response['data']['hits']:
                    print(f"First hit type: {type(json_response['data']['hits'][0])}")
                    print(f"First hit keys: {list(json_response['data']['hits'][0].keys()) if isinstance(json_response['data']['hits'][0], dict) else 'Not a dict'}")
        
        return json_response["data"]["hits"]
    
    def get_mutation_files(self, project_ids: List[str], genes: Optional[List[str]] = None) -> List[Dict]:
            """Get mutation annotation files for specified projects and genes"""
            filters = {
                "op": "and",
                "content": [
                    {
                        "op": "in",
                        "content": {
                            "field": "cases.project.project_id",
                            "value": project_ids
                        }
                    },
                    {
                        "op": "in",
                        "content": {
                            "field": "data_category",
                            "value": ["Simple Nucleotide Variation"]
                        }
                    },
                    {
                        "op": "in",
                        "content": {
                            "field": "data_type",
                            "value": ["Masked Somatic Mutation"]
                        }
                    },
                    {
                        "op": "in",
                        "content": {
                            "field": "experimental_strategy",
                            "value": ["WXS"]  # Whole Exome Sequencing
                        }
                    }
                ]
            }
            
            # Add gene filter if specified
            if genes:
                filters["content"].append({
                    "op": "in",
                    "content": {
                        "field": "ssms.gene.symbol",
                        "value": genes
                    }
                })
            
            endpoint = f"{self.base_url}/files"
            params = {
                "filters": json.dumps(filters),
                "expand": "cases.project,cases.submitter_id",
                "format": "json",
                "size": "2000"
            }
            
            response = requests.get(endpoint, params=params)
            response.raise_for_status()
            return response.json()["data"]["hits"]
    
    def get_clinical_data(self, project_ids: List[str]) -> pd.DataFrame:
        """Download clinical data for specified projects"""
        endpoint = f"{self.base_url}/cases"
        filters = {
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "project.project_id",
                        "value": project_ids
                    }
                }
            ]
        }
        
        params = {
            "filters": json.dumps(filters),
            "expand": "diagnoses,demographic,exposures,project",
            "format": "json",
            "size": "2000"
        }
        
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        
        cases = response.json()["data"]["hits"]
        clinical_data = []
        
        for case in cases:
            base_info = {
                "case_id": case["id"],
                "submitter_id": case["submitter_id"],
                "project_id": case["project"]["project_id"] if case.get("project") else None
            }
            
            # Add demographic data
            if case.get("demographic") and len(case["demographic"]) > 0:

                if type(case["demographic"]) is list:
                    demo = case["demographic"][0]
                else:
                    demo = case["demographic"]

                base_info.update({
                    "gender": demo.get("gender"),
                    "race": demo.get("race"),
                    "ethnicity": demo.get("ethnicity"),
                    "age_at_diagnosis": demo.get("age_at_diagnosis"),
                    "days_to_birth": demo.get("days_to_birth"),
                    "days_to_death": demo.get("days_to_death"),
                    "vital_status": demo.get("vital_status")
                })
            
            # Add diagnosis data
            if case.get("diagnoses") and len(case["diagnoses"]) > 0:
                if type(case["diagnoses"]) is list:
                    diag = case["diagnoses"][0]
                else:
                    diag = case["diagnoses"]

                base_info.update({
                    "primary_diagnosis": diag.get("primary_diagnosis"),
                    "tumor_stage": diag.get("tumor_stage"),
                    "tumor_grade": diag.get("tumor_grade"),
                    "tissue_or_organ_of_origin": diag.get("tissue_or_organ_of_origin"),
                    "site_of_resection_or_biopsy": diag.get("site_of_resection_or_biopsy"),
                    "days_to_diagnosis": diag.get("days_to_diagnosis"),
                    "age_at_diagnosis": diag.get("age_at_diagnosis"),
                    "morphology": diag.get("morphology"),
                    "primary_diagnosis_id": diag.get("id")
                })
            
            # Add exposure data if available
            if case.get("exposures") and len(case["exposures"]) > 0:
                if type(case["exposures"]) is list:
                    exp = case["exposures"][0]
                else:
                    exp = case["exposures"]

                base_info.update({
                    "cigarettes_per_day": exp.get("cigarettes_per_day"),
                    "years_smoked": exp.get("years_smoked"),
                    "pack_years_smoked": exp.get("pack_years_smoked"),
                    "alcohol_history": exp.get("alcohol_history")
                })
            
            clinical_data.append(base_info)
        
        print(f"Retrieved clinical data for {len(clinical_data)} cases")
        return pd.DataFrame(clinical_data)
    
    def download_file(self, file_id: str, filename: str) -> str:
        """Download a single file from GDC"""
        endpoint = f"{self.base_url}/data/{file_id}"
        
        filepath = self.output_dir / filename
        
        # Skip if file already exists
        if filepath.exists():
            print(f"File {filename} already exists, skipping...")
            return str(filepath)
        
        print(f"Downloading {filename}...")
        response = requests.get(endpoint, stream=True)
        response.raise_for_status()
        
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        return str(filepath)
    
    def parse_mutation_files(self, mutation_files: List[str], genes: Optional[List[str]] = None) -> pd.DataFrame:   
            """Parse downloaded mutation files and extract relevant mutations"""
            import gzip
            import zipfile
            
            all_mutations = []
            
            for file_path in mutation_files:
                print(f"Parsing {file_path}...")
                try:
                    # Check if file is compressed and read accordingly
                    df = None
                    
                    # Try to detect file type by reading first few bytes
                    with open(file_path, 'rb') as f:
                        header = f.read(10)
                    
                    if header.startswith(b'\x1f\x8b'):  # gzip magic number
                        print(f"  Detected gzip compressed file")
                        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                            df = pd.read_csv(f, sep='\t', low_memory=False, comment='#')
                    elif header.startswith(b'PK'):  # zip magic number
                        print(f"  Detected zip compressed file")
                        with zipfile.ZipFile(file_path, 'r') as zip_file:
                            # Get the first file in the zip
                            names = zip_file.namelist()
                            if names:
                                with zip_file.open(names[0]) as f:
                                    df = pd.read_csv(f, sep='\t', low_memory=False, comment='#')
                    else:
                        # Try as plain text file
                        print(f"  Treating as plain text file")
                        try:
                            df = pd.read_csv(file_path, sep='\t', low_memory=False, comment='#', encoding='utf-8')
                        except UnicodeDecodeError:
                            # Try with different encoding
                            print(f"  UTF-8 failed, trying latin-1 encoding")
                            df = pd.read_csv(file_path, sep='\t', low_memory=False, comment='#', encoding='latin-1')
                    
                    if df is None or df.empty:
                        print(f"  No data found in file")
                        continue
                    
                    print(f"  File has {len(df)} rows and {len(df.columns)} columns")
                    print(f"  Column names: {list(df.columns[:10])}...")  # Show first 10 columns
                    
                    # Filter by genes if specified
                    if genes:
                        if 'Hugo_Symbol' in df.columns:
                            initial_len = len(df)
                            df = df[df['Hugo_Symbol'].isin(genes)]
                            print(f"  Filtered by genes: {initial_len} -> {len(df)} rows")
                        elif 'Gene_Symbol' in df.columns:
                            initial_len = len(df)
                            df = df[df['Gene_Symbol'].isin(genes)]
                            print(f"  Filtered by genes (Gene_Symbol): {initial_len} -> {len(df)} rows")
                        else:
                            print(f"  Warning: No gene symbol column found for filtering")
                    
                    # Select relevant columns (be flexible with column names)
                    possible_cols = {
                        'Hugo_Symbol': ['Hugo_Symbol', 'Gene_Symbol', 'gene_symbol'],
                        'Variant_Classification': ['Variant_Classification', 'variant_classification', 'Mutation_Type'],
                        'Variant_Type': ['Variant_Type', 'variant_type'],
                        'Tumor_Sample_Barcode': ['Tumor_Sample_Barcode', 'Sample_ID', 'sample_id', 'Tumor_Sample_ID'],
                        'HGVSp': ['HGVSp', 'protein_change', 'Protein_Change'],
                        'HGVSc': ['HGVSc', 'cdna_change'],
                        'Chromosome': ['Chromosome', 'chr', 'chrom'],
                        'Start_Position': ['Start_Position', 'start_position', 'Start_position'],
                        'End_Position': ['End_Position', 'end_position', 'End_position'],
                        'Reference_Allele': ['Reference_Allele', 'ref', 'Ref'],
                        'Tumor_Seq_Allele2': ['Tumor_Seq_Allele2', 'alt', 'Alt', 'Tumor_Allele'],
                        'IMPACT': ['IMPACT', 'impact', 'Impact']
                    }
                    
                    selected_cols = {}
                    for standard_name, possible_names in possible_cols.items():
                        for col_name in possible_names:
                            if col_name in df.columns:
                                selected_cols[standard_name] = col_name
                                break
                    
                    print(f"  Found columns: {selected_cols}")
                    
                    if selected_cols:
                        # Create dataframe with standardized column names
                        df_filtered = df[list(selected_cols.values())].copy()
                        df_filtered.columns = list(selected_cols.keys())
                        
                        # Add source file info
                        df_filtered['source_file'] = os.path.basename(file_path)
                        
                        print(f"  Selected {len(df_filtered)} mutations with {len(df_filtered.columns)} columns")
                        all_mutations.append(df_filtered)
                    else:
                        print(f"  Warning: No recognized columns found")
                    
                except Exception as e:
                    print(f"Error parsing {file_path}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue
            
            if all_mutations:
                result_df = pd.concat(all_mutations, ignore_index=True)
                print(f"Total mutations parsed: {len(result_df)}")
                return result_df
            else:
                print("No mutations were successfully parsed")
                return pd.DataFrame()
    
    def create_patient_mutation_dataset(self, 
                                      project_ids: List[str], 
                                      genes: Optional[List[str]] = None,
                                      max_files: int = 50) -> Dict[str, pd.DataFrame]:
        """Create a comprehensive dataset of patients with gene mutations"""
        
        print("Getting project information...")
        projects = self.get_projects()
        selected_projects = [p for p in projects if p["id"] in project_ids]
        
        print(f"Selected projects: {[p['id'] for p in selected_projects]}")
        
        print("Getting clinical data...")
        clinical_df = self.get_clinical_data(project_ids)
        clinical_df.to_csv(self.output_dir / "clinical_data.csv", index=False)
        print(f"Downloaded clinical data for {len(clinical_df)} patients")
        
        print("Finding mutation files...")
        mutation_files = self.get_mutation_files(project_ids, genes)
        print(f"Found {len(mutation_files)} mutation files")
        
        # Debug: Check structure of first file
        if mutation_files:
            print(f"First file structure: {type(mutation_files[0])}")
            print(f"First file content: {mutation_files[0]}")
        
        # Limit files to avoid overwhelming downloads
        if len(mutation_files) > max_files:
            print(f"Limiting to first {max_files} files")
            mutation_files = mutation_files[:max_files]
        
        # Download mutation files
        downloaded_files = []
        for i, file_info in enumerate(mutation_files):
            try:
                # Handle both dict and string cases
                if isinstance(file_info, dict):
                    file_id = file_info['id']
                    filename = f"mutations_{i}_{file_id}.maf"
                else:
                    # If file_info is a string, it might be the file_id itself
                    file_id = str(file_info)
                    filename = f"mutations_{i}_{file_id}.maf"
                
                filepath = self.download_file(file_id, filename)
                downloaded_files.append(filepath)
                
                # Add delay to be respectful to the API
                time.sleep(1)
                
            except Exception as e:
                print(f"Error downloading file {file_info}: {e}")
                continue
        
        print("Parsing mutation data...")
        mutations_df = self.parse_mutation_files(downloaded_files, genes)
        
        if not mutations_df.empty:
            mutations_df.to_csv(self.output_dir / "mutations_data.csv", index=False)
            print(f"Parsed {len(mutations_df)} mutations")
        
        # Create summary dataset
        summary_data = self.create_summary_dataset(clinical_df, mutations_df)
        
        return {
            "clinical": clinical_df,
            "mutations": mutations_df,
            "summary": summary_data
        }
    
    def create_summary_dataset(self, clinical_df: pd.DataFrame, mutations_df: pd.DataFrame) -> pd.DataFrame:
        """Create a summary dataset combining clinical and mutation data"""
        if mutations_df.empty:
            return clinical_df
        
        # Extract patient IDs from tumor barcodes (first 12 characters)
        if 'Tumor_Sample_Barcode' in mutations_df.columns:
            mutations_df['patient_id'] = mutations_df['Tumor_Sample_Barcode'].str[:12]
            
            # Group mutations by patient
            patient_mutations = mutations_df.groupby('patient_id').agg({
                'Hugo_Symbol': lambda x: ';'.join(x.unique()),
                'Variant_Classification': lambda x: ';'.join(x.unique()),
                'Variant_Type': lambda x: ';'.join(x.unique())
            }).reset_index()
            
            patient_mutations.columns = ['patient_id', 'mutated_genes', 'variant_classifications', 'variant_types']
            
            # Add mutation counts
            mutation_counts = mutations_df.groupby('patient_id').size().reset_index(name='mutation_count')
            patient_mutations = patient_mutations.merge(mutation_counts, on='patient_id')
            
            # Merge with clinical data
            clinical_df['patient_id'] = clinical_df['submitter_id'].str[:12]
            summary_df = clinical_df.merge(patient_mutations, on='patient_id', how='left')
            
            # Fill NaN values for patients without mutations
            summary_df['mutation_count'] = summary_df['mutation_count'].fillna(0)
            summary_df['has_mutations'] = summary_df['mutation_count'] > 0
            
            summary_df.to_csv(self.output_dir / "patient_summary.csv", index=False)
            print(f"Created summary dataset with {len(summary_df)} patients")
            
            return summary_df
        
        return clinical_df

In [ ]:
#!/usr/bin/env python3
"""
TCGA Data Download Script for Gene Mutations
Downloads mutation data and clinical data from TCGA via GDC API
"""

import requests
import json
import pandas as pd
import os
import time
from typing import List, Dict, Optional
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from tqdm.notebook import tqdm

class TCGADownloader:
    def __init__(self, output_dir: str = "tcga_data"):
        self.base_url = "https://api.gdc.cancer.gov"
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
    def get_projects(self) -> List[Dict]:
        """Get all available TCGA projects"""
        endpoint = f"{self.base_url}/projects"
        params = {
            "filters": json.dumps({
                "op": "in",
                "content": {
                    "field": "program.name",
                    "value": ["TCGA"]
                }
            }),
            "format": "json",
            "size": "2000"
        }
        
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        
        # Debug the response structure
        json_response = response.json()
        print(f"Files API response keys: {json_response.keys()}")
        if 'data' in json_response:
            print(f"Data keys: {json_response['data'].keys()}")
            if 'hits' in json_response['data']:
                print(f"Number of hits: {len(json_response['data']['hits'])}")
                if json_response['data']['hits']:
                    print(f"First hit type: {type(json_response['data']['hits'][0])}")
                    print(f"First hit keys: {list(json_response['data']['hits'][0].keys()) if isinstance(json_response['data']['hits'][0], dict) else 'Not a dict'}")
        
        return json_response["data"]["hits"]
    
    def get_mutation_files(self, project_ids: List[str], genes: Optional[List[str]] = None) -> List[Dict]:
        """Get mutation annotation files for specified projects and genes"""
        filters = {
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "cases.project.project_id",
                        "value": project_ids
                    }
                },
                {
                    "op": "in",
                    "content": {
                        "field": "data_category",
                        "value": ["Simple Nucleotide Variation"]
                    }
                },
                {
                    "op": "in",
                    "content": {
                        "field": "data_type",
                        "value": ["Masked Somatic Mutation"]
                    }
                },
                {
                    "op": "in",
                    "content": {
                        "field": "experimental_strategy",
                        "value": ["WXS"]  # Whole Exome Sequencing
                    }
                }
            ]
        }
        
        # Add gene filter if specified
        if genes:
            filters["content"].append({
                "op": "in",
                "content": {
                    "field": "ssms.gene.symbol",
                    "value": genes
                }
            })
        
        endpoint = f"{self.base_url}/files"
        params = {
            "filters": json.dumps(filters),
            "expand": "cases.project,cases.submitter_id",
            "format": "json",
            "size": "2000"
        }
        
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        return response.json()["data"]["hits"]
    
    def get_clinical_data(self, project_ids: List[str]) -> pd.DataFrame:
        """Download clinical data for specified projects"""
        endpoint = f"{self.base_url}/cases"
        filters = {
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "project.project_id",
                        "value": project_ids
                    }
                }
            ]
        }
        
        params = {
            "filters": json.dumps(filters),
            "expand": "diagnoses,demographic,exposures,project",
            "format": "json",
            "size": "2000"
        }
        
        response = requests.get(endpoint, params=params)
        response.raise_for_status()
        
        cases = response.json()["data"]["hits"]
        clinical_data = []
        
        for case in cases:
            base_info = {
                "case_id": case["id"],
                "submitter_id": case["submitter_id"],
                "project_id": case["project"]["project_id"] if case.get("project") else None
            }
            
            # Add demographic data
            if case.get("demographic") and len(case["demographic"]) > 0:
                if type(case["demographic"]) is list:
                    demo = case["demographic"][0]
                else:
                    demo = case["demographic"]

                base_info.update({
                    "gender": demo.get("gender"),
                    "race": demo.get("race"),
                    "ethnicity": demo.get("ethnicity"),
                    "age_at_diagnosis": demo.get("age_at_diagnosis"),
                    "days_to_birth": demo.get("days_to_birth"),
                    "days_to_death": demo.get("days_to_death"),
                    "vital_status": demo.get("vital_status")
                })
            
            # Add diagnosis data
            if case.get("diagnoses") and len(case["diagnoses"]) > 0:
                diag = case["diagnoses"][0]
                base_info.update({
                    "primary_diagnosis": diag.get("primary_diagnosis"),
                    "tumor_stage": diag.get("tumor_stage"),
                    "tumor_grade": diag.get("tumor_grade"),
                    "tissue_or_organ_of_origin": diag.get("tissue_or_organ_of_origin"),
                    "site_of_resection_or_biopsy": diag.get("site_of_resection_or_biopsy"),
                    "days_to_diagnosis": diag.get("days_to_diagnosis"),
                    "age_at_diagnosis": diag.get("age_at_diagnosis"),
                    "morphology": diag.get("morphology"),
                    "primary_diagnosis_id": diag.get("id")
                })
            
            # Add exposure data if available
            if case.get("exposures") and len(case["exposures"]) > 0:
                exp = case["exposures"][0]
                base_info.update({
                    "cigarettes_per_day": exp.get("cigarettes_per_day"),
                    "years_smoked": exp.get("years_smoked"),
                    "pack_years_smoked": exp.get("pack_years_smoked"),
                    "alcohol_history": exp.get("alcohol_history")
                })
            
            clinical_data.append(base_info)
        
        print(f"Retrieved clinical data for {len(clinical_data)} cases")
        return pd.DataFrame(clinical_data)
    
    def download_file(self, file_id: str, filename: str) -> str:
        """Download a single file from GDC"""
        endpoint = f"{self.base_url}/data/{file_id}"
        
        filepath = self.output_dir / filename
        
        # Skip if file already exists
        if filepath.exists():
            print(f"File {filename} already exists, skipping...")
            return str(filepath)
        
        print(f"Downloading {filename}...")
        try:
            response = requests.get(endpoint, stream=True, timeout=300)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print(f"Successfully downloaded {filename}")
            return str(filepath)
            
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
            # Clean up partial file if it exists
            if filepath.exists():
                filepath.unlink()
            raise
    
    def download_files_parallel(self, file_info_list: List[Dict], max_workers: int = 4) -> List[str]:
        """Download multiple files in parallel"""
        downloaded_files = []
        failed_downloads = []
        
        def download_single_file(i, file_info):
            """Download a single file with error handling"""
            try:
                # Handle both dict and string cases
                if isinstance(file_info, dict):
                    file_id = file_info['id']
                    filename = f"mutations_{i}_{file_id}.maf"
                else:
                    # If file_info is a string, it might be the file_id itself
                    file_id = str(file_info)
                    filename = f"mutations_{i}_{file_id}.maf"
                
                filepath = self.download_file(file_id, filename)
                return filepath
                
            except Exception as e:
                print(f"Failed to download file {file_info}: {e}")
                return None
        
        print(f"Starting parallel download of {len(file_info_list)} files with {max_workers} workers...")
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all download tasks
            future_to_file = {
                executor.submit(download_single_file, i, file_info): (i, file_info) 
                for i, file_info in enumerate(file_info_list)
            }
            
            # Collect results as they complete
            for future in as_completed(future_to_file):
                i, file_info = future_to_file[future]
                try:
                    result = future.result()
                    if result:
                        downloaded_files.append(result)
                    else:
                        failed_downloads.append((i, file_info))
                        
                except Exception as e:
                    print(f"Exception in download task {i}: {e}")
                    failed_downloads.append((i, file_info))
        
        print(f"Download completed: {len(downloaded_files)} successful, {len(failed_downloads)} failed")
        
        if failed_downloads:
            print("Failed downloads:")
            for i, file_info in failed_downloads:
                print(f"  {i}: {file_info}")
        
        return downloaded_files
    
    def parse_mutation_files(self, mutation_files: List[str], genes: Optional[List[str]] = None) -> pd.DataFrame:
        """Parse downloaded mutation files and extract relevant mutations"""
        import gzip
        import zipfile
        
        all_mutations = []
        
        for file_path in mutation_files:
            print(f"Parsing {file_path}...")
            try:
                # Check if file is compressed and read accordingly
                df = None
                
                # Try to detect file type by reading first few bytes
                with open(file_path, 'rb') as f:
                    header = f.read(10)
                
                if header.startswith(b'\x1f\x8b'):  # gzip magic number
                    print(f"  Detected gzip compressed file")
                    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                        df = pd.read_csv(f, sep='\t', low_memory=False, comment='#')
                elif header.startswith(b'PK'):  # zip magic number
                    print(f"  Detected zip compressed file")
                    with zipfile.ZipFile(file_path, 'r') as zip_file:
                        # Get the first file in the zip
                        names = zip_file.namelist()
                        if names:
                            with zip_file.open(names[0]) as f:
                                df = pd.read_csv(f, sep='\t', low_memory=False, comment='#')
                else:
                    # Try as plain text file
                    print(f"  Treating as plain text file")
                    try:
                        df = pd.read_csv(file_path, sep='\t', low_memory=False, comment='#', encoding='utf-8')
                    except UnicodeDecodeError:
                        # Try with different encoding
                        print(f"  UTF-8 failed, trying latin-1 encoding")
                        df = pd.read_csv(file_path, sep='\t', low_memory=False, comment='#', encoding='latin-1')
                
                if df is None or df.empty:
                    print(f"  No data found in file")
                    continue
                
                print(f"  File has {len(df)} rows and {len(df.columns)} columns")
                print(f"  Column names: {list(df.columns[:10])}...")  # Show first 10 columns
                
                # Filter by genes if specified
                if genes:
                    if 'Hugo_Symbol' in df.columns:
                        initial_len = len(df)
                        df = df[df['Hugo_Symbol'].isin(genes)]
                        print(f"  Filtered by genes: {initial_len} -> {len(df)} rows")
                    elif 'Gene_Symbol' in df.columns:
                        initial_len = len(df)
                        df = df[df['Gene_Symbol'].isin(genes)]
                        print(f"  Filtered by genes (Gene_Symbol): {initial_len} -> {len(df)} rows")
                    else:
                        print(f"  Warning: No gene symbol column found for filtering")
                
                # Select relevant columns (be flexible with column names)
                possible_cols = {
                    'Hugo_Symbol': ['Hugo_Symbol', 'Gene_Symbol', 'gene_symbol'],
                    'Variant_Classification': ['Variant_Classification', 'variant_classification', 'Mutation_Type'],
                    'Variant_Type': ['Variant_Type', 'variant_type'],
                    'Tumor_Sample_Barcode': ['Tumor_Sample_Barcode', 'Sample_ID', 'sample_id', 'Tumor_Sample_ID'],
                    'HGVSp': ['HGVSp', 'protein_change', 'Protein_Change'],
                    'HGVSc': ['HGVSc', 'cdna_change'],
                    'Chromosome': ['Chromosome', 'chr', 'chrom'],
                    'Start_Position': ['Start_Position', 'start_position', 'Start_position'],
                    'End_Position': ['End_Position', 'end_position', 'End_position'],
                    'Reference_Allele': ['Reference_Allele', 'ref', 'Ref'],
                    'Tumor_Seq_Allele2': ['Tumor_Seq_Allele2', 'alt', 'Alt', 'Tumor_Allele'],
                    'IMPACT': ['IMPACT', 'impact', 'Impact']
                }
                
                selected_cols = {}
                for standard_name, possible_names in possible_cols.items():
                    for col_name in possible_names:
                        if col_name in df.columns:
                            selected_cols[standard_name] = col_name
                            break
                
                print(f"  Found columns: {selected_cols}")
                
                if selected_cols:
                    # Create dataframe with standardized column names
                    df_filtered = df[list(selected_cols.values())].copy()
                    df_filtered.columns = list(selected_cols.keys())
                    
                    # Add source file info
                    df_filtered['source_file'] = os.path.basename(file_path)
                    
                    print(f"  Selected {len(df_filtered)} mutations with {len(df_filtered.columns)} columns")
                    all_mutations.append(df_filtered)
                else:
                    print(f"  Warning: No recognized columns found")
                
            except Exception as e:
                print(f"Error parsing {file_path}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        if all_mutations:
            result_df = pd.concat(all_mutations, ignore_index=True)
            print(f"Total mutations parsed: {len(result_df)}")
            return result_df
        else:
            print("No mutations were successfully parsed")
            return pd.DataFrame()

    def create_patient_gene_matrix(self, clinical_df: pd.DataFrame, mutations_df: pd.DataFrame) -> pd.DataFrame:
        """Create a patient-gene matrix with clinical info and gene mutation data"""
        if mutations_df.empty:
            print("No mutation data available")
            return clinical_df
        
        print("Creating patient-gene mutation matrix...")
        
        # Extract patient IDs from tumor barcodes (first 12 characters)
        if 'Tumor_Sample_Barcode' in mutations_df.columns:
            mutations_df['patient_id'] = mutations_df['Tumor_Sample_Barcode'].str[:12]
        else:
            print("Error: No Tumor_Sample_Barcode column found")
            return clinical_df
        
        # Get all unique genes
        if 'Hugo_Symbol' in mutations_df.columns:
            all_genes = sorted(mutations_df['Hugo_Symbol'].unique())
            print(f"Found {len(all_genes)} unique genes")
        else:
            print("Error: No Hugo_Symbol column found")
            return clinical_df
        
        # Create patient ID for clinical data
        clinical_df['patient_id'] = clinical_df['submitter_id'].str[:12]
        
        # Start with clinical data
        patient_matrix = clinical_df.copy()
        
        # For each gene, create a column with mutation information
        for gene in tqdm(all_genes):
            # print(f"Processing gene: {gene}")
            
            # Get mutations for this gene
            gene_mutations = mutations_df[mutations_df['Hugo_Symbol'] == gene].copy()
            
            if gene_mutations.empty:
                continue
            
            # Group by patient and combine mutation information
            patient_gene_data = gene_mutations.groupby('patient_id').agg({
                'Variant_Classification': lambda x: ';'.join(x.unique()) if len(x) > 0 else '',
                'Variant_Type': lambda x: ';'.join(x.unique()) if len(x) > 0 else '',
                'HGVSp': lambda x: ';'.join([str(v) for v in x.unique() if pd.notna(v)]) if len(x) > 0 else '',
            }).reset_index()
            
            # Create a comprehensive mutation string for each patient-gene combination
            patient_gene_data[f'{gene}_mutation'] = patient_gene_data.apply(
                lambda row: f"{row['Variant_Classification']}|{row['Variant_Type']}|{row['HGVSp']}" 
                if row['Variant_Classification'] else '', axis=1
            )
            
            # Merge with patient matrix
            patient_matrix = patient_matrix.merge(
                patient_gene_data[['patient_id', f'{gene}_mutation']], 
                on='patient_id', 
                how='left'
            )
            
            # Fill NaN with empty string (no mutation)
            patient_matrix[f'{gene}_mutation'] = patient_matrix[f'{gene}_mutation'].fillna('')
        
        # Create binary mutation columns (1 if mutated, 0 if not)
        print("Creating binary mutation columns...")
        for gene in all_genes:
            mutation_col = f'{gene}_mutation'
            binary_col = f'{gene}_mutated'
            
            if mutation_col in patient_matrix.columns:
                patient_matrix[binary_col] = (patient_matrix[mutation_col] != '').astype(int)
        
        # Add summary statistics
        mutation_cols = [col for col in patient_matrix.columns if col.endswith('_mutated')]
        patient_matrix['total_mutations'] = patient_matrix[mutation_cols].sum(axis=1)
        patient_matrix['mutation_burden'] = patient_matrix['total_mutations'] / len(all_genes)
        
        print(f"Created patient-gene matrix with {len(patient_matrix)} patients and {len(all_genes)} genes")
        print(f"Matrix shape: {patient_matrix.shape}")
        
        return patient_matrix
    
    def create_patient_mutation_dataset(self, 
                                      project_ids: List[str], 
                                      genes: Optional[List[str]] = None,
                                      max_files: int = 50,
                                      max_workers: int = 4) -> Dict[str, pd.DataFrame]:
        """Create a comprehensive dataset of patients with gene mutations"""
        
        print("Getting project information...")
        projects = self.get_projects()
        selected_projects = [p for p in projects if p["id"] in project_ids]
        
        print(f"Selected projects: {[p['id'] for p in selected_projects]}")
        
        print("Getting clinical data...")
        clinical_df = self.get_clinical_data(project_ids)
        clinical_df.to_csv(self.output_dir / "clinical_data.csv", index=False)
        print(f"Downloaded clinical data for {len(clinical_df)} patients")
        
        print("Finding mutation files...")
        mutation_files = self.get_mutation_files(project_ids, genes)
        print(f"Found {len(mutation_files)} mutation files")
        
        # Debug: Check structure of first file
        if mutation_files:
            print(f"First file structure: {type(mutation_files[0])}")
            print(f"First file content: {mutation_files[0]}")
        
        # Limit files to avoid overwhelming downloads
        if len(mutation_files) > max_files:
            print(f"Limiting to first {max_files} files")
            mutation_files = mutation_files[:max_files]
        
        # Download mutation files in parallel
        print(f"Downloading {len(mutation_files)} files using {max_workers} parallel workers...")
        downloaded_files = self.download_files_parallel(mutation_files, max_workers=max_workers)
        
        print("Parsing mutation data...")
        mutations_df = self.parse_mutation_files(downloaded_files, genes)
        
        if not mutations_df.empty:
            mutations_df.to_csv(self.output_dir / "mutations_data.csv", index=False)
            print(f"Parsed {len(mutations_df)} mutations")
        
        # Create patient-gene mutation matrix
        patient_gene_matrix = self.create_patient_gene_matrix(clinical_df, mutations_df)
        patient_gene_matrix.to_csv(self.output_dir / "patient_gene_matrix.csv", index=False)
        print(f"Saved patient-gene matrix to patient_gene_matrix.csv")
        
        return {
            "clinical": clinical_df,
            "mutations": mutations_df,
            "patient_gene_matrix": patient_gene_matrix
        }

In [ ]:
projects = ["TCGA-BRCA"]
output = "/home/cc/PHD/dglframework/cptac/tgca_datasets"

downloader = TCGADownloader(output)

print(f"Starting TCGA data download...")
print(f"Projects: {projects}")
print(f"Genes: all")
print(f"Output directory: {output}")

datasets = downloader.create_patient_mutation_dataset(
    project_ids=projects,
    genes=None,
    max_files=5000,
     max_workers=32
)

print("\nDownload completed!")
print(f"Clinical data: {len(datasets['clinical'])} patients")
print(f"Mutation data: {len(datasets['mutations'])} mutations")

print(f"Files saved to: {output}")